# SVM Model — Campus Waste Forecasting

This notebook trains a Support Vector Machine (SVM) regressor for waste forecasting. SVM finds the optimal hyperplane that best fits the data while maximizing the margin. Unlike tree-based methods, SVM requires feature scaling to perform well. The kernel trick allows SVM to capture non-linear relationships.

## 0. Imports

All required libraries are imported here. Scikit-learn provides the SVR (Support Vector Regression) implementation. StandardScaler normalizes features to zero mean and unit variance, which is essential for SVM performance.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

## 1. Hyperparameters

SVM has three key hyperparameters: C (regularization strength), epsilon (tube width for no penalty), and kernel-specific parameters like gamma. The RBF (Radial Basis Function) kernel is most commonly used for non-linear regression.

In [ ]:
# ── Forecast Settings ─────────────────────────────────────────────────────────
FORECAST_HORIZON = 7          # Days to predict ahead
TEST_DAYS = 30                # Days held out for final test evaluation
VALIDATION_SPLITS = 3         # Number of CV folds for hyperparameter tuning

# ── SVM Default Hyperparameters ───────────────────────────────────────────────
SVM_PARAMS = {
    'kernel': 'rbf',           # Kernel type: 'rbf', 'linear', 'poly'
    'C': 100,                  # Regularization (higher = less regularization)
    'epsilon': 0.1,            # Epsilon-tube width (no penalty within tube)
    'gamma': 'scale',          # Kernel coefficient ('scale', 'auto', or float)
    'cache_size': 500          # Cache size in MB (increase for large datasets)
}

# ── Grid Search Settings ──────────────────────────────────────────────────────
ENABLE_GRID_SEARCH = True      # Set to False to use default parameters
GRID_SEARCH_PARAMS = {
    'svr__C': [1, 10, 100, 1000],
    'svr__epsilon': [0.01, 0.1, 0.5],
    'svr__gamma': ['scale', 'auto', 0.1, 0.01]
}

# ── Feature Selection ─────────────────────────────────────────────────────────
EXCLUDE_COLS = ['date', 'Canteen_Section', 'Waste_Weight_kg']

# ── Data Path ─────────────────────────────────────────────────────────────────
DATA_PATH = 'data/waste_features_xgb.csv'

print("Hyperparameters loaded.")
print(f"Grid search: {'Enabled' if ENABLE_GRID_SEARCH else 'Disabled'}")

## 2. Helper Functions

Modular functions for data preparation, scaling, model training, and evaluation. SVM requires special handling due to mandatory feature scaling.

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load the feature-engineered dataset and parse dates."""
    df = pd.read_csv(path)
    df['date'] = pd.to_datetime(df['date'])
    return df


def prepare_features(df: pd.DataFrame, section: str = None, exclude_cols: list = None):
    """
    Prepare feature matrix X and target y.
    
    Args:
        df: Dataset with all features
        section: If provided, filter to this section only
        exclude_cols: Columns to exclude from features
    
    Returns:
        X: Feature DataFrame
        y: Target Series
        dates: Date Series for indexing
    """
    if section:
        data = df[df['Canteen_Section'] == section].copy()
    else:
        data = df.copy()
    
    data = data.sort_values('date').reset_index(drop=True)
    
    exclude = exclude_cols or []
    feature_cols = [c for c in data.columns if c not in exclude]
    
    X = data[feature_cols]
    y = data['Waste_Weight_kg']
    dates = data['date']
    
    return X, y, dates


def temporal_split(X, y, dates, test_days: int):
    """
    Split data temporally. The most recent 'test_days' become the test set.
    """
    n = len(X)
    split_idx = n - test_days
    
    X_train = X.iloc[:split_idx]
    X_test = X.iloc[split_idx:]
    y_train = y.iloc[:split_idx]
    y_test = y.iloc[split_idx:]
    train_dates = dates.iloc[:split_idx]
    test_dates = dates.iloc[split_idx:]
    
    return X_train, X_test, y_train, y_test, train_dates, test_dates


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Compute evaluation metrics for regression.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}


def baseline_moving_average(y_train, y_test, window: int = 7) -> np.ndarray:
    """
    Compute a simple moving average baseline.
    """
    all_y = pd.concat([y_train, y_test], ignore_index=True)
    train_len = len(y_train)
    
    predictions = []
    for i in range(len(y_test)):
        idx = train_len + i
        start_idx = max(0, idx - window)
        predictions.append(all_y.iloc[start_idx:idx].mean())
    
    return np.array(predictions)


def create_svm_pipeline(svm_params: dict):
    """
    Create a scikit-learn pipeline with StandardScaler and SVR.
    
    A pipeline ensures scaling is applied consistently during training and prediction,
    and prevents data leakage from test set into scaling parameters.
    """
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(**svm_params))
    ])
    return pipeline


def train_svm_with_cv(X_train, y_train, svm_params: dict, param_grid: dict, cv_splits: int):
    """
    Train SVM with time-series cross-validation and optional grid search.
    
    Returns:
        best_pipeline: Trained pipeline with best parameters
        best_params: Dictionary of best parameters
        scaler: Fitted StandardScaler for later use
    """
    tscv = TimeSeriesSplit(n_splits=cv_splits)
    
    pipeline = create_svm_pipeline(svm_params)
    
    if param_grid:
        print(f"Running grid search with {cv_splits}-fold time series CV...")
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=tscv,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X_train, y_train)
        
        best_pipeline = grid_search.best_estimator_
        best_params = grid_search.best_params_
        print(f"Best parameters: {best_params}")
        print(f"Best CV RMSE: {-grid_search.best_score_:.4f}")
    else:
        print("Training with default parameters...")
        pipeline.fit(X_train, y_train)
        best_pipeline = pipeline
        best_params = svm_params
    
    return best_pipeline, best_params


def print_metrics_comparison(baseline_metrics: dict, model_metrics: dict, model_name: str = "SVM"):
    """Print a formatted comparison of baseline vs model metrics."""
    print("\n" + "=" * 55)
    print(f"{'Metric':<10} {'Baseline':>12} {model_name:>12} {'Improvement':>15}")
    print("=" * 55)
    
    for metric in ['RMSE', 'MAE', 'MAPE', 'R2']:
        base_val = baseline_metrics[metric]
        model_val = model_metrics[metric]
        
        if metric == 'R2':
            improvement = model_val - base_val
            imp_str = f"+{improvement:.4f}" if improvement > 0 else f"{improvement:.4f}"
        else:
            improvement = ((base_val - model_val) / base_val) * 100 if base_val != 0 else 0
            imp_str = f"{improvement:+.1f}%"
        
        print(f"{metric:<10} {base_val:>12.4f} {model_val:>12.4f} {imp_str:>15}")
    
    print("=" * 55)

## 3. Load and Explore Data

In [ ]:
df = load_data(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Sections: {sorted(df['Canteen_Section'].unique())}")
print(f"\nFeature columns ({len(df.columns) - 3} features):")
feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]
print(feature_cols)
df.head()

## 4. Feature Scaling Importance

SVM is sensitive to feature scales. Features with larger magnitudes can dominate the distance calculations. We use StandardScaler to transform features to zero mean and unit variance. The scaler is fit only on training data to prevent data leakage.

In [ ]:
# Demonstrate scale differences before scaling
sample_section = 'A'
X_sample, _, _ = prepare_features(df, sample_section, EXCLUDE_COLS)

print("Feature Scale Summary (before scaling):")
print("="*60)
scale_summary = pd.DataFrame({
    'mean': X_sample.mean(),
    'std': X_sample.std(),
    'min': X_sample.min(),
    'max': X_sample.max()
}).round(2)

# Show features with most extreme scales
print("\nFeatures with largest scale differences:")
print(scale_summary.nlargest(5, 'std')[['mean', 'std', 'min', 'max']])
print("\nFeatures with smallest scale:")
print(scale_summary.nsmallest(5, 'std')[['mean', 'std', 'min', 'max']])

## 5. Train and Evaluate SVM (All Sections)

We train separate SVM models for each canteen section. The pipeline handles scaling automatically. Grid search explores different combinations of C, epsilon, and gamma.

In [ ]:
sections = sorted(df['Canteen_Section'].unique())
results = {}

for section in sections:
    print(f"\n{'='*60}")
    print(f"SECTION {section}")
    print(f"{'='*60}")
    
    # Prepare features
    X, y, dates = prepare_features(df, section, EXCLUDE_COLS)
    print(f"Total observations: {len(X)}, Features: {X.shape[1]}")
    
    # Temporal split
    X_train, X_test, y_train, y_test, train_dates, test_dates = temporal_split(
        X, y, dates, TEST_DAYS
    )
    print(f"Train: {len(X_train)} | Test: {len(X_test)}")
    
    # Baseline
    baseline_pred = baseline_moving_average(y_train, y_test, window=7)
    baseline_metrics = compute_metrics(y_test.values, baseline_pred)
    print(f"\nBaseline (7-day MA): RMSE={baseline_metrics['RMSE']:.4f}")
    
    # Train SVM
    param_grid = GRID_SEARCH_PARAMS if ENABLE_GRID_SEARCH else None
    pipeline, best_params = train_svm_with_cv(
        X_train, y_train, SVM_PARAMS, param_grid, VALIDATION_SPLITS
    )
    
    # Predict on test set
    svm_pred = pipeline.predict(X_test)
    
    # Evaluate
    svm_metrics = compute_metrics(y_test.values, svm_pred)
    
    print_metrics_comparison(baseline_metrics, svm_metrics, "SVM")
    
    # Store results
    results[section] = {
        'pipeline': pipeline,
        'best_params': best_params,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'test_dates': test_dates,
        'predictions': svm_pred,
        'baseline_pred': baseline_pred,
        'baseline_metrics': baseline_metrics,
        'svm_metrics': svm_metrics,
        'feature_names': list(X.columns)
    }

## 6. Visualize Forecasts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, section in enumerate(sections):
    ax = axes[idx]
    res = results[section]
    
    test_dates = res['test_dates']
    actual = res['y_test'].values
    
    ax.plot(test_dates, actual, 'b-', label='Actual', linewidth=1.5)
    ax.plot(test_dates, res['predictions'], 'r--', label='SVM', linewidth=1.5)
    ax.plot(test_dates, res['baseline_pred'], 'g:', label='Baseline (MA)', linewidth=1.5)
    
    ax.set_title(f"Section {section} — Test Set Forecast")
    ax.set_xlabel('Date')
    ax.set_ylabel('Waste (kg)')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Support Vector Analysis

SVM identifies support vectors, which are the data points closest to the decision boundary. These points define the model. Examining the number and distribution of support vectors gives insight into model complexity.

In [ ]:
print("Support Vector Analysis")
print("=" * 55)

for section in sections:
    res = results[section]
    svr_model = res['pipeline'].named_steps['svr']
    
    n_support = svr_model.n_support_
    n_train = len(res['X_train'])
    support_ratio = n_support / n_train * 100
    
    print(f"\nSection {section}:")
    print(f"  Support vectors: {n_support} ({support_ratio:.1f}% of training data)")
    print(f"  Training samples: {n_train}")
    
# Note: A high support vector ratio may indicate:
# - Complex decision boundary needed
# - Potential overfitting (if very high)
# - Need for different C or epsilon values

## 8. Seven-Day Ahead Forecast

Like XGBoost, SVM requires recursive prediction for multi-step forecasting. The pipeline handles scaling automatically during prediction.

In [ ]:
def forecast_7_days_svm(pipeline, last_row: pd.Series, feature_names: list,
                        df_section: pd.DataFrame) -> pd.DataFrame:
    """
    Generate a 7-day ahead forecast using recursive prediction.
    
    The pipeline handles scaling automatically.
    
    Args:
        pipeline: Trained sklearn pipeline (scaler + SVR)
        last_row: Last row of features from training data
        feature_names: List of feature column names
        df_section: Original section data for computing rolling features
    
    Returns:
        DataFrame with date and predicted waste
    """
    predictions = []
    current_features = last_row.copy()
    last_date = df_section['date'].max()
    
    # Get recent history for rolling calculations
    recent_waste = list(df_section['Waste_Weight_kg'].tail(28).values)
    
    for day in range(1, 8):
        future_date = last_date + pd.Timedelta(days=day)
        
        # Update calendar features
        current_features['weekday'] = future_date.dayofweek
        current_features['month'] = future_date.month
        current_features['quarter'] = future_date.quarter
        current_features['day_of_year'] = future_date.dayofyear
        current_features['week_of_year'] = future_date.isocalendar().week
        current_features['year'] = future_date.year
        
        # Update cyclical features
        current_features['sin_weekday'] = np.sin(2 * np.pi * future_date.dayofweek / 7)
        current_features['cos_weekday'] = np.cos(2 * np.pi * future_date.dayofweek / 7)
        current_features['sin_month'] = np.sin(2 * np.pi * future_date.month / 12)
        current_features['cos_month'] = np.cos(2 * np.pi * future_date.month / 12)
        current_features['sin_doy'] = np.sin(2 * np.pi * future_date.dayofyear / 365)
        current_features['cos_doy'] = np.cos(2 * np.pi * future_date.dayofyear / 365)
        
        # Update lag features
        if 'lag_1' in feature_names:
            current_features['lag_1'] = recent_waste[-1]
        if 'lag_7' in feature_names:
            current_features['lag_7'] = recent_waste[-7] if len(recent_waste) >= 7 else recent_waste[0]
        if 'lag_14' in feature_names:
            current_features['lag_14'] = recent_waste[-14] if len(recent_waste) >= 14 else recent_waste[0]
        if 'lag_28' in feature_names:
            current_features['lag_28'] = recent_waste[-28] if len(recent_waste) >= 28 else recent_waste[0]
        
        # Update rolling features
        if 'rolling_mean_7' in feature_names:
            current_features['rolling_mean_7'] = np.mean(recent_waste[-7:])
        if 'rolling_mean_14' in feature_names:
            current_features['rolling_mean_14'] = np.mean(recent_waste[-14:])
        if 'rolling_std_7' in feature_names:
            current_features['rolling_std_7'] = np.std(recent_waste[-7:])
        if 'rolling_max_7' in feature_names:
            current_features['rolling_max_7'] = np.max(recent_waste[-7:])
        
        # Assume normal day for future
        current_features['is_holiday'] = 0
        current_features['is_special_day'] = 1 if future_date.dayofweek >= 5 else 0
        current_features['is_weekend'] = 1 if future_date.dayofweek >= 5 else 0
        
        # Predict (pipeline handles scaling)
        X_pred = pd.DataFrame([current_features[feature_names]])
        pred = pipeline.predict(X_pred)[0]
        pred = max(0, pred)  # Ensure non-negative
        
        predictions.append({
            'date': future_date,
            'predicted_waste_kg': pred
        })
        
        # Update recent waste for next iteration
        recent_waste.append(pred)
    
    return pd.DataFrame(predictions)

In [ ]:
print("7-Day Ahead Forecast (from last date in data)")
print("=" * 70)

forecasts_7day = {}

for section in sections:
    res = results[section]
    
    # Get section data
    section_df = df[df['Canteen_Section'] == section].sort_values('date')
    
    # Get last row of features
    last_row = res['X_test'].iloc[-1].copy()
    
    forecast_df = forecast_7_days_svm(
        res['pipeline'], last_row, res['feature_names'], section_df
    )
    forecasts_7day[section] = forecast_df
    
    print(f"\nSection {section}:")
    print(forecast_df.to_string(index=False))

## 9. Kernel Comparison (Optional)

Different kernels capture different relationship types. Linear kernel assumes linear relationships. RBF (default) can capture non-linear patterns. Polynomial kernel captures polynomial relationships of specified degree.

In [ ]:
# Compare different kernels for one section
sample_section = 'A'
X, y, dates = prepare_features(df, sample_section, EXCLUDE_COLS)
X_train, X_test, y_train, y_test, _, _ = temporal_split(X, y, dates, TEST_DAYS)

kernels = ['rbf', 'linear', 'poly']
kernel_results = []

print(f"\nKernel Comparison for Section {sample_section}:")
print("=" * 55)

for kernel in kernels:
    params = SVM_PARAMS.copy()
    params['kernel'] = kernel
    
    pipeline = create_svm_pipeline(params)
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    
    metrics = compute_metrics(y_test.values, pred)
    kernel_results.append({
        'Kernel': kernel,
        'RMSE': metrics['RMSE'],
        'MAE': metrics['MAE'],
        'R2': metrics['R2']
    })

kernel_df = pd.DataFrame(kernel_results)
print(kernel_df.round(4).to_string(index=False))

## 10. Results Summary

In [ ]:
print("\n" + "=" * 80)
print("OVERALL RESULTS SUMMARY")
print("=" * 80)

summary_data = []
for section in sections:
    res = results[section]
    summary_data.append({
        'Section': section,
        'Baseline_RMSE': res['baseline_metrics']['RMSE'],
        'SVM_RMSE': res['svm_metrics']['RMSE'],
        'Baseline_MAE': res['baseline_metrics']['MAE'],
        'SVM_MAE': res['svm_metrics']['MAE'],
        'Baseline_R2': res['baseline_metrics']['R2'],
        'SVM_R2': res['svm_metrics']['R2'],
        'RMSE_Improvement_%': ((res['baseline_metrics']['RMSE'] - res['svm_metrics']['RMSE']) / 
                               res['baseline_metrics']['RMSE'] * 100)
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.round(4).to_string(index=False))

print(f"\nAverage RMSE Improvement: {summary_df['RMSE_Improvement_%'].mean():.2f}%")
print(f"Average SVM R2: {summary_df['SVM_R2'].mean():.4f}")

## 11. Save Model

In [ ]:
import pickle

os.makedirs('models', exist_ok=True)

for section in sections:
    model_path = f'models/svm_section_{section}.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump({
            'pipeline': results[section]['pipeline'],
            'best_params': results[section]['best_params'],
            'feature_names': results[section]['feature_names']
        }, f)
    print(f"Saved: {model_path}")

print("\nAll SVM models saved.")